In [45]:
import logging
import os
from typing import Optional

import h5py
import hist
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score

hep.style.use(hep.style.ROOT)
logging.basicConfig(level=logging.INFO)

PLOT_DIR = os.path.join(os.getcwd(), "multi_roc_curves")
os.makedirs(PLOT_DIR, exist_ok=True)

BASE_TPR = np.linspace(0., 1., 1000)
NTOPS = 2
NJETS = 3*NTOPS + 4
NFJETS = NTOPS + 1

FILES = {
    "tt": {"eval": "/storage/af/user/tsievert/topNet/h5s/tt_hadronic_noBQ.h5", "test": "/storage/af/user/tsievert/topNet/h5s/fjTag_testing.h5"},
    "qcd": {"eval": "/storage/af/user/tsievert/topNet/h5s/qcd_4j_Alltesting_noBQeval.h5", "test": "/storage/af/user/tsievert/topNet/h5s/qcd_4j_wTargets_Alltesting.h5"},
    "t": {"eval": "/storage/af/user/tsievert/topNet/h5s/tt_semileptonic_Alltesting_noBQeval.h5", "test": "/storage/af/user/tsievert/topNet/h5s/ttbar_semileptonic/ttbar_semileptonic.h5"}
    # "4t": "",
}


In [44]:
LOADED_FILES = {label: {filetype: h5py.File(filepath) for filetype, filepath in filepaths.items()} for label, filepaths in FILES.items()}
RECO_CLASSES = list(LOADED_FILES[next(iter(LOADED_FILES))]["eval"]["SpecialKey.Targets"].keys())
SCORES = [item for item in LOADED_FILES[next(iter(LOADED_FILES))]["eval"]["SpecialKey.Targets"][next(iter(RECO_CLASSES))].keys() if "probability" in item]

ROC_CURVES = {
    # tt performance
    "Correct tt vs. QCD": {'sig': [(True, LOADED_FILES["tt"])], 'bkg': [(False, LOADED_FILES["qcd"])]},
    "Correct tt vs. Incorrect tt": {'sig': [(True, LOADED_FILES["tt"])], 'bkg': [(False, LOADED_FILES["tt"])]},
    # tt vs. t comparison (should be random if theyre equal performance)
    # "tt vs. t": {'sig': [(True, LOADED_FILES["tt"])], 'bkg': [(True, LOADED_FILES["t"])]},
    # t performance
    "Correct t vs. QCD": {'sig': [(True, LOADED_FILES["t"])], 'bkg': [(False, LOADED_FILES["qcd"])]},
    "Correct t vs. Incorrect t": {'sig': [(True, LOADED_FILES["t"])], 'bkg': [(False, LOADED_FILES["t"])]},
    # Everything
    "Correct tt and t vs. QCD and Incorrect tt and t": {'sig': [(True, LOADED_FILES["tt"]), (True, LOADED_FILES["t"])], 'bkg': [(False, LOADED_FILES["qcd"]), (False, LOADED_FILES["tt"]), (False, LOADED_FILES["t"])]},
}

In [35]:
def n_alpha(string: str):
    return len([c for c in string if c.isalpha()])

def find_correct_assignments(reco_class: str, dataset: dict[str, str]):
    assignments = [item for item in dataset["eval"]["SpecialKey.Targets"][reco_class].keys() if "probability" not in item]
    correct_assignments = np.ones_like(dataset["eval"]["SpecialKey.Targets"][reco_class][assignments[0]], dtype=bool)
    for assignment in assignments:
        correct_assignments = np.logical_and(
            correct_assignments,
            dataset["eval"]["SpecialKey.Targets"][reco_class][assignment][:] == dataset["test"]["TARGETS"][reco_class][assignment][:]
            if n_alpha(assignment) == 1 else 
            dataset["eval"]["SpecialKey.Targets"][reco_class][assignment][:] == (dataset["test"]["TARGETS"][reco_class][assignment][:] + NJETS)
        )
    return correct_assignments

def get_sigORbkg_pred_and_truth(reco_class: str, score: str, datasets: list[tuple[bool, dict[str, str]]]):
    preds, truths, = [], []
    for correct_assignment, dataset in datasets:
        correct_assignment_mask = (find_correct_assignments(reco_class, dataset) == correct_assignment)
        all_preds = dataset["eval"]["SpecialKey.Targets"][reco_class][score][:]
        all_truths = dataset["test"]["TARGETS"][reco_class]["MASK"][:]
        preds.append(all_preds[correct_assignment_mask]); truths.append(all_truths[correct_assignment_mask])
    return np.concatenate(preds), np.concatenate(truths)

def get_sigANDbkg_preds_and_truths(reco_class: str, score: str, roc_curve_datasets: dict[str, list[tuple[bool, dict[str, str]]]]):
    signal_preds, signal_truths = get_sigORbkg_pred_and_truth(reco_class, score, roc_curve_datasets['sig'])
    background_preds, background_truths = get_sigORbkg_pred_and_truth(reco_class, score, roc_curve_datasets['bkg'])

    return np.concatenate((signal_preds, background_preds)), np.concatenate((signal_truths, background_truths))

In [ ]:
def plot_roc(fprs: list[np.ndarray], tprs: list[np.ndarray], labels: list[str], title: Optional[str]=None, xlabel: Optional[str]=None, ylabel: Optional[str]=None, save: Optional[str]=None, show: bool=False):
    if xlabel is None: xlabel = 'Bkg Eff.'
    if ylabel is None: ylabel = 'Sig Eff.'

    plt.figure(figsize=(10, 8))
    for fpr, tpr, label in zip(fprs, tprs, labels):
        plt.plot(fpr, tpr, label=label)
    plt.legend()
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xscale('log')
    plt.yscale('log')
    if title is not None: plt.title(title)
    if save is not None: plt.savefig(os.path.join(PLOT_DIR, save))
    if show: plt.show()
    else: plt.close()


In [46]:
for roc_curve_label, roc_curve_datasets in ROC_CURVES.items():
    for reco_class in RECO_CLASSES:
        try:
            fprs, tprs, labels = [], [], []
            for score in SCORES:
                preds, truths = get_sigANDbkg_preds_and_truths(reco_class, score, roc_curve_datasets)
                fpr, tpr, thresholds = roc_curve(truths, preds)
                auc = roc_auc_score(truths, preds)
                fprinterp = np.interp(BASE_TPR, tpr, fpr)
                thresholdsinterp = np.interp(BASE_TPR, tpr, thresholds)
                fprs.append(fprinterp); tprs.append(BASE_TPR), labels.append(score.replace('_probability', '')+f' - AUC = {auc:.3f}')
            plot_roc(fprs, tprs, labels, title=roc_curve_label+' - '+reco_class, save='_'.join(roc_curve_label.split(' '))+'_'+reco_class+'.png')
        except:
            print(f'Error creating ROC curve for \'{roc_curve_label}\' with class \'{reco_class}\'')
            continue
            

/storage/af/user/tsievert/topNet/spatop_venv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


Error creating ROC curve for 'Correct t vs. QCD' with class 'FBt2'


/storage/af/user/tsievert/topNet/spatop_venv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


Error creating ROC curve for 'Correct t vs. QCD' with class 'FRt2'


/storage/af/user/tsievert/topNet/spatop_venv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


...

Error creating ROC curve for 'Correct t vs. Incorrect t' with class 'FBt2'


/storage/af/user/tsievert/topNet/spatop_venv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


Error creating ROC curve for 'Correct t vs. Incorrect t' with class 'FRt2'


/storage/af/user/tsievert/topNet/spatop_venv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1183: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


Error creating ROC curve for 'Correct t vs. Incorrect t' with class 'SRqqt2'


In [48]:
for files in LOADED_FILES.values(): 
    for file in files.values(): file.close()